<a href="https://colab.research.google.com/github/hiyashah1098/Achievement-Benchmarker-/blob/main/Achievement_Benchmarker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Achievement Benchmarker**

My app helps a small indie studio decide whether to include Steam achievements in an upcoming game by comparing ratings, average playtime, and estimated ownership for games in the same genre with and without achievements.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/MIST4600/


**compare_ratings_by_achievements Function**

In [4]:
def compare_ratings_by_achievements(games, genre):
    genre_games = games[
        games["genres"].str.contains(genre, case=False, na=False)
    ].copy()

    genre_games["achievement_group"] = "Has achievements"
    genre_games.loc[
        genre_games["achievements"] == 0, "achievement_group"
    ] = "No achievements"

    comparison = (
        genre_games.groupby("achievement_group")["pct_pos_total"]
        .mean()
        .reset_index()
        .rename(columns={"pct_pos_total": "average_positive_review_pct"})
    )

    comparison["average_positive_review_pct"] = (
        comparison["average_positive_review_pct"].round(2)
    )

    return comparison


my_function = compare_ratings_by_achievements
my_result = my_function(games, "Action")
print(my_result)

  achievement_group  average_positive_review_pct
0  Has achievements                        84.90
1   No achievements                        81.52


**compare_playtime_by_achievements Function**

In [3]:
def compare_playtime_by_achievements(games, genre):
    genre_games = games[
        games["genres"].str.contains(genre, case=False, na=False)
    ].copy()

    genre_games["achievement_group"] = "Has achievements"
    genre_games.loc[
        genre_games["achievements"] == 0, "achievement_group"
    ] = "No achievements"

    comparison = (
        genre_games.groupby("achievement_group")["avg_playtime_hours"]
        .mean()
        .reset_index()
        .rename(columns={"avg_playtime_hours": "average_playtime_hours"})
    )

    comparison["average_playtime_hours"] = (
        comparison["average_playtime_hours"].round(2)
    )

    return comparison


my_function = compare_playtime_by_achievements
my_result = my_function(games, "Action")
print(my_result)

  achievement_group  average_playtime_hours
0  Has achievements                   26.97
1   No achievements                   25.20


**compare_ownership_by_achievements Function**

In [5]:
def compare_ownership_by_achievements(games, genre):
    genre_games = games[
        games["genres"].str.contains(genre, case=False, na=False)
    ].copy()

    genre_games["achievement_group"] = "Has achievements"
    genre_games.loc[
        genre_games["achievements"] == 0, "achievement_group"
    ] = "No achievements"

    comparison = (
        genre_games.groupby("achievement_group")["owners_low"]
        .mean()
        .reset_index()
        .rename(columns={"owners_low": "average_owners_lower_bound"})
    )

    comparison["average_owners_lower_bound"] = (
        comparison["average_owners_lower_bound"].round(2)
    )

    return comparison


my_function = compare_ownership_by_achievements
my_result = my_function(games, "Action")
print(my_result)

  achievement_group  average_owners_lower_bound
0  Has achievements                  3643738.32
1   No achievements                  4679347.83


**GROQ API Key Integration**

In [10]:
%pip install -q groq

from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Reply with exactly: API connection works."}]
)

print(response.choices[0].message.content)

API connection works.


****Use Groq to Compare Ratings by Achievement Status and Play Mode****

In [11]:
import pandas as pd

def label_play_mode(description):
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Label this game Single-player, Multiplayer, Both, or Unclear. "
                    "Reply with only one label."
                )
            },
            {"role": "user", "content": description}
        ]
    )

    return response.choices[0].message.content.strip()


def compare_ratings_by_mode_and_achievements(games, genre):
    genre_games = games[
        games["genres"].str.contains(genre, case=False, na=False)
    ].dropna(
        subset=["short_description", "pct_pos_total", "achievements"]
    ).copy()

    genre_games["achievement_group"] = "Has achievements"
    genre_games.loc[
        genre_games["achievements"] == 0, "achievement_group"
    ] = "No achievements"

    no_achievements = genre_games[
        genre_games["achievement_group"] == "No achievements"
    ].head(5)

    has_achievements = genre_games[
        genre_games["achievement_group"] == "Has achievements"
    ].head(5)

    sample = pd.concat([no_achievements, has_achievements])

    sample["play_mode"] = sample["short_description"].apply(label_play_mode)

    comparison = (
        sample.groupby(["achievement_group", "play_mode"])["pct_pos_total"]
        .mean()
        .reset_index()
        .rename(columns={
            "pct_pos_total": "average_positive_review_pct"
        })
    )

    comparison["average_positive_review_pct"] = (
        comparison["average_positive_review_pct"].round(2)
    )

    return comparison


results = compare_ratings_by_mode_and_achievements(games, "Action")
print(results)

  achievement_group    play_mode  average_positive_review_pct
0  Has achievements         Both                        92.00
1  Has achievements  Multiplayer                        78.00
2   No achievements         Both                        94.00
3   No achievements  Multiplayer                        89.25
